# Linear Regression from Scratch

**Interview-focused reference notebook** -- NumPy implementations, key intuitions, complexity analysis.

## Core Intuition

Linear regression fits a hyperplane $\hat{y} = X\mathbf{w} + b$ that minimizes the sum of squared residuals.

**Geometric interpretation:** We project the target vector $\mathbf{y}$ onto the column space of $X$. The residual $\mathbf{y} - X\mathbf{w}$ is orthogonal to every column of $X$, which gives us the normal equation.

**Key formulas:**
- Loss: $L(\mathbf{w}) = \frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|^2$
- Gradient: $\nabla L = \frac{1}{n}X^T(X\mathbf{w} - \mathbf{y})$
- Normal equation: $\mathbf{w}^* = (X^TX)^{-1}X^T\mathbf{y}$

**Complexity:**
- Normal equation: $O(nd^2 + d^3)$ -- forming $X^TX$ is $O(nd^2)$, inverting is $O(d^3)$
- Gradient descent: $O(ndi)$ where $i$ = number of iterations

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
import matplotlib.cm as cm
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Synthetic Data Generation

In [ ]:
def generate_regression_data(n=200, d=5, noise=0.5, seed=42):
    """Generate synthetic regression data with known true weights."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, d)
    true_w = np.array([3.0, -1.5, 2.0, 0.0, 0.0])[:d]  # last features are irrelevant
    true_b = 1.0
    y = X @ true_w + true_b + noise * rng.randn(n)
    return X, y, true_w, true_b

X, y, true_w, true_b = generate_regression_data()
print(f"X shape: {X.shape}, y shape: {y.shape}")
print(f"True weights: {true_w}, True bias: {true_b}")

## 2. Normal Equation (Closed-Form)

Derive by setting $\nabla L = 0$:

$$X^TX\mathbf{w} = X^T\mathbf{y} \implies \mathbf{w}^* = (X^TX)^{-1}X^T\mathbf{y}$$

**When does this fail?** (Interview classic)
1. $X^TX$ is singular (rank-deficient) -- more features than samples, or collinear features
2. $d$ is very large -- $O(d^3)$ inversion is prohibitive
3. Data doesn't fit in memory -- need iterative methods

In [ ]:
class LinearRegressionNormal:
    """Linear regression via normal equation."""
    
    def fit(self, X, y):
        # Add bias column
        X_b = np.column_stack([np.ones(X.shape[0]), X])
        # Normal equation: w = (X^T X)^{-1} X^T y
        # Use np.linalg.solve for numerical stability (avoids explicit inverse)
        self.w_ = np.linalg.solve(X_b.T @ X_b, X_b.T @ y)
        self.bias_ = self.w_[0]
        self.coef_ = self.w_[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.bias_

model_normal = LinearRegressionNormal().fit(X, y)
print(f"Normal eq weights: {model_normal.coef_.round(3)}")
print(f"Normal eq bias:    {model_normal.bias_:.3f}")
print(f"True weights:      {true_w}")
print(f"True bias:         {true_b}")

## 3. Gradient Descent Solutions

Three flavors:
- **Batch GD**: uses all $n$ samples per step. Stable but slow for large $n$.
- **Stochastic GD (SGD)**: uses 1 sample per step. Noisy but fast. Can escape shallow local minima.
- **Mini-batch GD**: uses $B$ samples per step. Best of both worlds. Standard in deep learning.

In [ ]:
class LinearRegressionGD:
    """Linear regression via gradient descent (batch, SGD, mini-batch)."""
    
    def __init__(self, lr=0.01, n_iters=1000, method='batch', batch_size=32):
        self.lr = lr
        self.n_iters = n_iters
        self.method = method  # 'batch', 'sgd', 'mini-batch'
        self.batch_size = batch_size
        self.loss_history = []
        self.w_history = []  # for visualization
    
    def _mse_loss(self, X, y, w):
        residuals = X @ w - y
        return 0.5 * np.mean(residuals ** 2)
    
    def _gradient(self, X, y, w):
        """Gradient: (1/n) X^T (Xw - y)"""
        n = X.shape[0]
        return (1.0 / n) * X.T @ (X @ w - y)
    
    def fit(self, X, y):
        n, d = X.shape
        X_b = np.column_stack([np.ones(n), X])
        w = np.zeros(d + 1)
        
        for i in range(self.n_iters):
            if self.method == 'batch':
                grad = self._gradient(X_b, y, w)
            elif self.method == 'sgd':
                idx = np.random.randint(n)
                grad = self._gradient(X_b[idx:idx+1], y[idx:idx+1], w)
            elif self.method == 'mini-batch':
                indices = np.random.choice(n, self.batch_size, replace=False)
                grad = self._gradient(X_b[indices], y[indices], w)
            
            w -= self.lr * grad
            self.loss_history.append(self._mse_loss(X_b, y, w))
            if i % 50 == 0:
                self.w_history.append(w.copy())
        
        self.w_ = w
        self.bias_ = w[0]
        self.coef_ = w[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.bias_

# Train with all three methods
model_batch = LinearRegressionGD(lr=0.05, n_iters=500, method='batch').fit(X, y)
model_sgd = LinearRegressionGD(lr=0.01, n_iters=500, method='sgd').fit(X, y)
model_mini = LinearRegressionGD(lr=0.03, n_iters=500, method='mini-batch', batch_size=32).fit(X, y)

print("Batch GD weights: ", model_batch.coef_.round(3))
print("SGD weights:      ", model_sgd.coef_.round(3))
print("Mini-batch weights:", model_mini.coef_.round(3))
print("True weights:     ", true_w)

## 4. Loss Convergence Comparison

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
ax.plot(model_batch.loss_history, label='Batch GD', linewidth=2)
ax.plot(model_sgd.loss_history, label='SGD', alpha=0.7, linewidth=1)
ax.plot(model_mini.loss_history, label='Mini-batch GD', alpha=0.8, linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE Loss')
ax.set_title('Gradient Descent Convergence Comparison')
ax.legend()
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Loss Landscape Visualization (2D Case)

For a 1-feature model, the loss surface is a 2D paraboloid over $(w, b)$.

In [ ]:
# Generate simple 1D data for visualization
X_1d = np.random.randn(100, 1) * 2
y_1d = 3.0 * X_1d.ravel() + 1.0 + 0.5 * np.random.randn(100)

# Compute loss landscape
w_range = np.linspace(-1, 6, 100)
b_range = np.linspace(-3, 5, 100)
W, B = np.meshgrid(w_range, b_range)
Loss = np.zeros_like(W)
for i in range(W.shape[0]):
    for j in range(W.shape[1]):
        pred = X_1d.ravel() * W[i, j] + B[i, j]
        Loss[i, j] = 0.5 * np.mean((pred - y_1d) ** 2)

# Run GD on 1D data and track path
model_1d = LinearRegressionGD(lr=0.05, n_iters=200, method='batch')
model_1d.fit(X_1d, y_1d)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Contour plot with GD path
ax = axes[0]
cs = ax.contour(W, B, Loss, levels=30, cmap='viridis')
ax.clabel(cs, inline=True, fontsize=8)
path = np.array(model_1d.w_history)
if len(path) > 0:
    ax.plot(path[:, 1], path[:, 0], 'r.-', markersize=8, linewidth=2, label='GD path')
    ax.plot(path[0, 1], path[0, 0], 'go', markersize=12, label='Start')
    ax.plot(path[-1, 1], path[-1, 0], 'r*', markersize=15, label='End')
ax.set_xlabel('w (slope)')
ax.set_ylabel('b (intercept)')
ax.set_title('Loss Landscape + GD Path')
ax.legend()

# 3D surface
ax2 = fig.add_subplot(122, projection='3d')
ax2.plot_surface(W, B, Loss, cmap='viridis', alpha=0.7)
ax2.set_xlabel('w')
ax2.set_ylabel('b')
ax2.set_zlabel('Loss')
ax2.set_title('Loss Surface (Convex Paraboloid)')
axes[1].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Regularized Regression

### Why regularize?
- Prevent overfitting (reduce variance at the cost of some bias)
- Handle multicollinearity
- Feature selection (L1)

**Ridge (L2):** $L = \frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda\|\mathbf{w}\|_2^2$
- Closed form: $\mathbf{w}^* = (X^TX + \lambda I)^{-1}X^T\mathbf{y}$
- Shrinks coefficients toward zero but never exactly to zero

**Lasso (L1):** $L = \frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda\|\mathbf{w}\|_1$
- No closed form (not differentiable at 0)
- Use coordinate descent with soft-thresholding
- **Produces sparse solutions** -- sets some coefficients exactly to zero

**ElasticNet:** $L = \frac{1}{2n}\|X\mathbf{w} - \mathbf{y}\|^2 + \lambda_1\|\mathbf{w}\|_1 + \lambda_2\|\mathbf{w}\|_2^2$

### Interview: Why does L1 give sparsity but L2 doesn't?
The L1 penalty has corners at the axes. The contours of the loss function are ellipses. The intersection of ellipses with the L1 diamond is much more likely to occur at a corner (where some coordinate = 0) than the intersection with the L2 circle.

In [ ]:
class RidgeRegression:
    """Ridge regression (L2 regularization) -- closed-form solution."""
    
    def __init__(self, alpha=1.0):
        self.alpha = alpha
    
    def fit(self, X, y):
        n, d = X.shape
        X_b = np.column_stack([np.ones(n), X])
        # (X^T X + alpha * I)^{-1} X^T y  (don't regularize bias)
        reg_matrix = self.alpha * np.eye(d + 1)
        reg_matrix[0, 0] = 0  # don't regularize bias term
        self.w_ = np.linalg.solve(X_b.T @ X_b + reg_matrix, X_b.T @ y)
        self.bias_ = self.w_[0]
        self.coef_ = self.w_[1:]
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.bias_


class LassoRegression:
    """Lasso regression (L1) via coordinate descent with soft-thresholding."""
    
    def __init__(self, alpha=1.0, n_iters=1000, tol=1e-6):
        self.alpha = alpha
        self.n_iters = n_iters
        self.tol = tol
    
    @staticmethod
    def _soft_threshold(x, threshold):
        """Soft-thresholding operator: sign(x) * max(|x| - threshold, 0)"""
        return np.sign(x) * np.maximum(np.abs(x) - threshold, 0)
    
    def fit(self, X, y):
        n, d = X.shape
        # Standardize for coordinate descent (important for Lasso)
        self.X_mean_ = X.mean(axis=0)
        self.X_std_ = X.std(axis=0) + 1e-8
        self.y_mean_ = y.mean()
        X_s = (X - self.X_mean_) / self.X_std_
        y_c = y - self.y_mean_
        
        w = np.zeros(d)
        
        for iteration in range(self.n_iters):
            w_old = w.copy()
            for j in range(d):
                # Residual without feature j's contribution
                r_j = y_c - X_s @ w + X_s[:, j] * w[j]
                # Coordinate update with soft-thresholding
                rho_j = X_s[:, j] @ r_j / n
                w[j] = self._soft_threshold(rho_j, self.alpha)
            
            if np.max(np.abs(w - w_old)) < self.tol:
                break
        
        # Convert back to original scale
        self.coef_ = w / self.X_std_
        self.bias_ = self.y_mean_ - self.X_mean_ @ self.coef_
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.bias_


class ElasticNet:
    """ElasticNet via coordinate descent."""
    
    def __init__(self, alpha=1.0, l1_ratio=0.5, n_iters=1000, tol=1e-6):
        self.alpha = alpha
        self.l1_ratio = l1_ratio  # 1.0 = pure Lasso, 0.0 = pure Ridge
        self.n_iters = n_iters
        self.tol = tol
    
    def fit(self, X, y):
        n, d = X.shape
        self.X_mean_ = X.mean(axis=0)
        self.X_std_ = X.std(axis=0) + 1e-8
        self.y_mean_ = y.mean()
        X_s = (X - self.X_mean_) / self.X_std_
        y_c = y - self.y_mean_
        
        l1_pen = self.alpha * self.l1_ratio
        l2_pen = self.alpha * (1 - self.l1_ratio)
        
        w = np.zeros(d)
        for iteration in range(self.n_iters):
            w_old = w.copy()
            for j in range(d):
                r_j = y_c - X_s @ w + X_s[:, j] * w[j]
                rho_j = X_s[:, j] @ r_j / n
                w[j] = np.sign(rho_j) * max(abs(rho_j) - l1_pen, 0) / (1 + l2_pen)
            if np.max(np.abs(w - w_old)) < self.tol:
                break
        
        self.coef_ = w / self.X_std_
        self.bias_ = self.y_mean_ - self.X_mean_ @ self.coef_
        return self
    
    def predict(self, X):
        return X @ self.coef_ + self.bias_

# Test all models
ridge = RidgeRegression(alpha=1.0).fit(X, y)
lasso = LassoRegression(alpha=0.1).fit(X, y)
enet = ElasticNet(alpha=0.1, l1_ratio=0.5).fit(X, y)

print("True weights:     ", true_w)
print("OLS weights:      ", model_normal.coef_.round(3))
print("Ridge weights:    ", ridge.coef_.round(3))
print("Lasso weights:    ", lasso.coef_.round(3))
print("ElasticNet weights:", enet.coef_.round(3))
print("\nNote: Lasso drives irrelevant features (indices 3,4) closer to zero.")

## 7. Regularization Path -- Coefficient Shrinkage Visualization

In [ ]:
alphas = np.logspace(-3, 2, 50)

ridge_coefs = []
lasso_coefs = []

for a in alphas:
    r = RidgeRegression(alpha=a).fit(X, y)
    ridge_coefs.append(r.coef_)
    l = LassoRegression(alpha=a * 0.01).fit(X, y)  # scale alpha for comparable range
    lasso_coefs.append(l.coef_)

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for j in range(X.shape[1]):
    axes[0].plot(alphas, ridge_coefs[:, j], label=f'w{j} (true={true_w[j]:.1f})')
axes[0].set_xscale('log')
axes[0].set_xlabel('Regularization strength (alpha)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Ridge (L2) Regularization Path')
axes[0].legend(fontsize=9)
axes[0].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[0].grid(True, alpha=0.3)

for j in range(X.shape[1]):
    axes[1].plot(alphas, lasso_coefs[:, j], label=f'w{j} (true={true_w[j]:.1f})')
axes[1].set_xscale('log')
axes[1].set_xlabel('Regularization strength (alpha)')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso (L1) Regularization Path')
axes[1].legend(fontsize=9)
axes[1].axhline(y=0, color='k', linestyle='--', alpha=0.3)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Key observation: L2 shrinks all coefficients smoothly; L1 drives some exactly to zero (sparsity).")

## 8. L1 vs L2 Geometry -- Why L1 Gives Sparsity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# L1 constraint region (diamond)
theta = np.linspace(0, 2*np.pi, 200)
l1_x = np.cos(theta)
l1_y = np.sin(theta)
# Transform to L1 ball: |x| + |y| = 1
l1_x_ball = np.sign(np.cos(theta)) * np.abs(np.cos(theta))
l1_y_ball = np.sign(np.sin(theta)) * np.abs(np.sin(theta))
# Proper diamond
diamond_x = [1, 0, -1, 0, 1]
diamond_y = [0, 1, 0, -1, 0]

# Loss contours (ellipses centered at some w*)
w_star = np.array([0.8, 0.3])  # unconstrained optimum
w1_range = np.linspace(-2, 2, 200)
w2_range = np.linspace(-2, 2, 200)
W1, W2 = np.meshgrid(w1_range, w2_range)
Loss_contour = (W1 - w_star[0])**2 + 2*(W2 - w_star[1])**2

for ax, title, constraint_x, constraint_y in [
    (axes[0], 'L1 (Lasso) -- touches at corner (sparse!)', diamond_x, diamond_y),
    (axes[1], 'L2 (Ridge) -- touches on circle (not sparse)', np.cos(theta), np.sin(theta))
]:
    ax.contour(W1, W2, Loss_contour, levels=15, cmap='Blues', alpha=0.6)
    ax.fill(constraint_x, constraint_y, alpha=0.3, color='red', label='Constraint region')
    ax.plot(constraint_x, constraint_y, 'r-', linewidth=2)
    ax.plot(*w_star, 'k*', markersize=15, label=f'Unconstrained opt ({w_star[0]}, {w_star[1]})')
    ax.set_xlim(-2, 2)
    ax.set_ylim(-2, 2)
    ax.set_xlabel('w1')
    ax.set_ylabel('w2')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.set_aspect('equal')
    ax.axhline(y=0, color='k', alpha=0.2)
    ax.axvline(x=0, color='k', alpha=0.2)

plt.tight_layout()
plt.show()

## 9. Validation Against sklearn

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet as SkElasticNet
from sklearn.metrics import mean_squared_error, r2_score

# Split data
X_train, X_test = X[:160], X[160:]
y_train, y_test = y[:160], y[160:]

# Sklearn models
sk_lr = LinearRegression().fit(X_train, y_train)
sk_ridge = Ridge(alpha=1.0).fit(X_train, y_train)
sk_lasso = Lasso(alpha=0.1).fit(X_train, y_train)

# Our models
my_lr = LinearRegressionNormal().fit(X_train, y_train)
my_ridge = RidgeRegression(alpha=1.0).fit(X_train, y_train)
my_lasso = LassoRegression(alpha=0.1).fit(X_train, y_train)

print("Model Comparison (test MSE):")
print(f"{'Model':<20} {'Ours':>10} {'sklearn':>10}")
print("-" * 42)
for name, ours, theirs in [
    ('OLS', my_lr, sk_lr),
    ('Ridge', my_ridge, sk_ridge),
    ('Lasso', my_lasso, sk_lasso),
]:
    mse_ours = mean_squared_error(y_test, ours.predict(X_test))
    mse_theirs = mean_squared_error(y_test, theirs.predict(X_test))
    print(f"{name:<20} {mse_ours:>10.4f} {mse_theirs:>10.4f}")

print("\nCoefficient Comparison:")
print(f"{'Model':<15} {'Ours':>40} {'sklearn':>40}")
print("-" * 97)
for name, ours, theirs in [
    ('OLS', my_lr, sk_lr),
    ('Ridge', my_ridge, sk_ridge),
    ('Lasso', my_lasso, sk_lasso),
]:
    print(f"{name:<15} {str(ours.coef_.round(4)):>40} {str(theirs.coef_.round(4)):>40}")

## 10. Bias-Variance Tradeoff Demo

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Generate 1D data with nonlinear pattern
np.random.seed(42)
X_bv = np.sort(np.random.uniform(-3, 3, 30)).reshape(-1, 1)
y_bv = np.sin(X_bv.ravel()) + 0.3 * np.random.randn(30)
X_plot = np.linspace(-3, 3, 200).reshape(-1, 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
degrees = [1, 4, 15]
titles = ['Underfitting (high bias)', 'Good fit', 'Overfitting (high variance)']

for ax, deg, title in zip(axes, degrees, titles):
    poly = PolynomialFeatures(deg)
    X_poly = poly.fit_transform(X_bv)
    X_poly_plot = poly.transform(X_plot)
    
    model = LinearRegressionNormal().fit(X_poly[:, 1:], y_bv)  # skip bias col from poly
    y_pred = model.predict(X_poly_plot[:, 1:])
    
    ax.scatter(X_bv, y_bv, c='blue', s=30, alpha=0.7, label='Data')
    ax.plot(X_plot, y_pred, 'r-', linewidth=2, label=f'Degree {deg}')
    ax.plot(X_plot, np.sin(X_plot), 'g--', alpha=0.5, label='True function')
    ax.set_title(title)
    ax.set_ylim(-2.5, 2.5)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| When does closed-form fail? | $d$ too large ($O(d^3)$), $X^TX$ singular, data too large for memory |
| L1 vs L2 -- why sparsity? | L1 diamond has corners on axes; loss ellipses more likely to touch there |
| Bias-variance tradeoff? | More model complexity = less bias, more variance. Regularization increases bias, decreases variance. |
| GD vs SGD vs Mini-batch? | Batch: stable, slow. SGD: noisy, fast, can escape local minima. Mini-batch: practical sweet spot. |
| How to choose $\lambda$? | Cross-validation. Plot validation error vs $\lambda$, pick the elbow. |
| Assumptions of linear regression? | Linearity, independence, homoscedasticity, normality of residuals (for inference) |
| $R^2$ interpretation? | Fraction of variance explained. $R^2 = 1 - \frac{SS_{res}}{SS_{tot}}$ |